# Career Extraction — `Llama 3.3 70B`

Extracts all career spells (start_year, end_year, position) from raw CV text.
One JSON array per CV, evaluated with temporal fuzzy matching (±1 year tolerance).

## 1. Imports & config

In [ ]:
import json
import re
import time

import pandas as pd
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv(override=True)

from corex_eval import evaluate, load_inputs

from openai import OpenAI
LMSTUDIO_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")

MODEL_ID    = "meta/llama-3.3-70b"
TEMPERATURE = 0
MAX_TOKENS  = 2048

print(f"Model: {MODEL_ID}")

## 2. Load inputs

In [ ]:
inputs_df = load_inputs(task="extraction")
print(f"{len(inputs_df)} CVs in test set")
inputs_df.head()

## 3. System prompt

In [ ]:
SYSTEM_PROMPT = """\
You are an expert data extractor. Given a CV in any language, extract ALL career positions and return them as a JSON array.

For each position include:
- "start_year" : integer year the position started, or null if not found
- "end_year"   : integer year the position ended, or null if ongoing/not found
- "position"   : job title and organisation as written in the CV (keep original language)

Rules:
- Include every position mentioned, including part-time, board memberships, advisory roles
- If only one year is mentioned for a position, use it as start_year
- Do not infer or guess years that are not explicitly stated
- Respond with ONLY the JSON array — no explanation, no markdown fences

Example output:
[{"start_year": 1990, "end_year": 1995, "position": "Minister of Finance, Government of Belgium"},
 {"start_year": 1995, "end_year": null,  "position": "Member of Parliament"}]
"""
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

## 4. Run inference

In [ ]:
def parse_spells(text):
    """Extract JSON array of career spells from model response."""
    text = text.strip()
    # Strip markdown fences
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)
    # Find first [...] block
    m = re.search(r'\[[\s\S]*\]', text)
    if m:
        try:
            parsed = json.loads(m.group())
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    return []

rows = []
for _, row in tqdm(inputs_df.iterrows(), total=len(inputs_df), desc='Extracting'):
    case_id = str(row["case_id"])
    cv_text = str(row["cv_local"] or "")
    user_msg = f"CV text:\n\n{cv_text}"

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {case_id}: {e}")
        raw = "[]"
        time.sleep(1.0)

    rows.append({
        "case_id": case_id,
        "career":  parse_spells(raw),
    })

predictions_df = pd.DataFrame(rows)
# Quick sanity check
n_spells = predictions_df['career'].apply(len)
print(f'Avg spells per CV: {n_spells.mean():.1f}, max: {n_spells.max()}')
predictions_df.head()

## 5. Save predictions (cache)

In [ ]:
import pickle
with open('predictions_cache.pkl', 'wb') as f:
    pickle.dump(predictions_df, f)
print('Saved predictions_cache.pkl')

## 6. Evaluate & submit

In [ ]:
results = evaluate(
    predictions_df,
    task="extraction",
    variable="career",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/extraction/lmstudio_llama33_70b_career/config.yaml",
)
pd.Series({
    k: v for k, v in results.items()
    if k not in ("per_case",)
})

## 7. Inspect per-case results

In [ ]:
per_case = pd.DataFrame(results['per_case']).T
per_case = per_case.astype(float, errors='ignore')
per_case.sort_values('f1', ascending=True).head(20)